# 00 — Raw Data Sanity Check

Quick inspection of all five source files before any cleaning. Goal: understand schemas, spot surprises (weird headers, missing values, unit inconsistencies), and inform the cleaning plan in `01_data_ingestion.ipynb`.

| # | File | Source |
|---|---|---|
| 1 | `Global-Integrated-Power-March-2026-II.xlsx` | GEM GIPT |
| 2 | `Global-Nuclear-Power-Tracker-September-2025.xlsx` | GEM GNPT |
| 3 | `WEO2025_AnnexA_Free_Dataset_World.csv` | IEA WEO 2025 (global) |
| 4 | `WEO2025_AnnexA_Free_Dataset_Regions.csv` | IEA WEO 2025 (regional) |
| 5 | `Statistical Review of World Energy Narrow format.csv` | Energy Institute 2025 |

In [3]:
import pandas as pd
import numpy as np
from pathlib import Path

RAW = Path('../data/raw')

def peek(df, title=''):
    """Print a compact summary of a DataFrame."""
    print(f'=== {title} ===')
    print(f'Shape: {df.shape[0]:,} rows × {df.shape[1]} cols')
    print()
    null_pct = (df.isnull().mean() * 100).round(1)
    col_summary = pd.DataFrame({
        'dtype': df.dtypes,
        'null_%': null_pct,
        'n_unique': df.nunique(),
        'sample': df.iloc[0] if len(df) > 0 else None
    })
    display(col_summary)
    print()
    print('--- First 3 rows ---')
    display(df.head(3))
    print()

---
## 1. GEM — Global Integrated Power Tracker (GIPT)
`Global-Integrated-Power-March-2026-II.xlsx`

Unit-level power plants worldwide. Used for the full energy mix analysis and the bottom-up renewable buildout model.

In [4]:
gipt_path = RAW / 'Global-Integrated-Power-March-2026-II.xlsx'

# Check available sheets first
xl = pd.ExcelFile(gipt_path)
print('Sheets:', xl.sheet_names)

Sheets: ['About', 'Power facilities', 'Regions, area, and countries']


In [6]:
# Load one of the sheets
gipt = pd.read_excel(gipt_path, sheet_name='Power facilities')
peek(gipt, 'GEM GIPT')

=== GEM GIPT ===
Shape: 182,428 rows × 52 cols



,dtype,null_%,n_unique,sample
Type,str,0.0,8,bioenergy
Country/area,str,0.0,226,Guatemala
Subregion,str,0.0,17,Latin America and the Caribbean
Region,str,0.0,5,Americas
Plant / Project name,str,0.0,145109,Palo Gordo power station
Unit / Phase name,object,3.9,7399,2
Plant / Project name (local),object,61.7,51126,Planta cogeneradora Palo Gordo
Plant / Project name (other),object,82.1,20213,Ingenio Palo Gordo
Capacity (MW),float64,0.0,3476,46.0
Status,str,0.0,10,operating



--- First 3 rows ---


,Type,Country/area,Subregion,Region,Plant / Project name,Unit / Phase name,Plant / Project name (local),Plant / Project name (other),Capacity (MW),Status,...,Latitude,Longitude,Location accuracy,City,"Local area (taluk, county)","Major area (prefecture, district)","Subnational unit (state, province)",GEM location ID,GEM unit/phase ID,GEM.Wiki URL
0,bioenergy,Guatemala,Latin America and the Caribbean,Americas,Palo Gordo power station,2,Planta cogeneradora Palo Gordo,Ingenio Palo Gordo,46.0,operating,...,14.4880,-91.3986,exact,NaN,San Antonio Suchitepéquez,NaN,Suchitepéquez,L100000102016,G100000104857,https://www.gem.wiki/Palo_Gordo_power_station
1,bioenergy,Guatemala,Latin America and the Caribbean,Americas,Magdalena Sugar Mill power station,6,"Planta cogeneradora Magdalena, Planta De Vapor...",Finca Buganvilla,62.0,operating,...,14.1216,-90.9272,exact,NaN,La Democracia,NaN,Escuintla,L100000102012,G100000106560,https://www.gem.wiki/Magdalena_Sugar_Mill_powe...
2,bioenergy,Guatemala,Latin America and the Caribbean,Americas,Magdalena Sugar Mill power station,7,"Planta cogeneradora Magdalena, Planta De Vapor...",Finca Buganvilla,62.0,operating,...,14.1216,-90.9272,exact,NaN,La Democracia,NaN,Escuintla,L100000102012,G100000106561,https://www.gem.wiki/Magdalena_Sugar_Mill_powe...


So the first sheet is simply an introduction page, and the last sheet is a lookup table mapping countries to their GEM-defined subregions and regions.

Sheet index 1 stores the main data we will use. 

In [7]:
# Key fields to confirm: fuel type labels, status values, capacity units, coordinate columns
for col in ['Fuel', 'Fuel 1', 'Technology', 'Status', 'Capacity (MW)', 'Latitude', 'Longitude']:
    if col in gipt.columns:
        print(f'\n--- {col} ---')
        print(gipt[col].value_counts().head(20))


--- Technology ---
Technology
PV                           79156
Onshore                      32701
Assumed PV                   24500
subcritical                   8210
combined cycle                5250
gas turbine                   5172
unknown                       3695
conventional storage          3360
steam turbine                 2856
supercritical                 2114
ultra-supercritical           1769
run-of-river                  1710
pumped storage                1045
pressurized water reactor      941
Offshore hard mount            918
internal combustion            799
Offshore mount unknown         728
Offshore floating              439
unknown type                   337
Unknown                        303
Name: count, dtype: int64

--- Status ---
Status
operating                   127724
pre-construction             21886
announced                     7556
cancelled                     7128
construction                  6173
retired                       4251
cancelled 

So for cleaning this dataset in the next notebook, we will need to do the following:

- load sheet index 1 
- Merge "Assumed PV" with "PV"
- Map the inferred statuses

---
## 2. GEM — Global Nuclear Power Tracker (GNPT)
`Global-Nuclear-Power-Tracker-September-2025.xlsx`

Nuclear-specific unit data. Used for the nuclear pipeline analysis; status progression dates needed to derive historical delivery rates.

In [ ]:
gnpt_path = RAW / 'Global-Nuclear-Power-Tracker-September-2025.xlsx'

xl_gnpt = pd.ExcelFile(gnpt_path)
print('Sheets:', xl_gnpt.sheet_names)

In [ ]:
gnpt = pd.read_excel(gnpt_path, sheet_name=0)
peek(gnpt, 'GEM GNPT')

In [ ]:
# Key fields: status, planned/announced capacity, date columns for delivery-rate calculation
for col in ['Status', 'Capacity (MW)', 'Country', 'Start Year', 'Announced', 'Construction Start']:
    if col in gnpt.columns:
        print(f'\n--- {col} ---')
        print(gnpt[col].value_counts().head(20))

# Show all date-like column names
date_cols = [c for c in gnpt.columns if any(kw in c.lower() for kw in ['year', 'date', 'start', 'retire', 'open', 'close', 'commission'])]
print('\nDate-like columns:', date_cols)

---
## 3. IEA WEO 2025 — World (Global)
`WEO2025_AnnexA_Free_Dataset_World.csv`

Top-down global electricity demand projections to 2050 across three scenarios. Anchor for the demand gap model.

In [ ]:
iea_world = pd.read_csv(RAW / 'WEO2025_AnnexA_Free_Dataset_World.csv')
peek(iea_world, 'IEA WEO World')

In [ ]:
# Confirm scenarios, variables, and units
for col in ['Scenario', 'Category', 'Sub-Category', 'Flow', 'Unit', 'Year']:
    if col in iea_world.columns:
        print(f'\n--- {col} ---')
        print(iea_world[col].value_counts().head(20))

---
## 4. IEA WEO 2025 — Regions
`WEO2025_AnnexA_Free_Dataset_Regions.csv`

Same projections broken out by IEA region. Critical for the regional analysis (South/Southeast Asia, Sub-Saharan Africa, etc.).

In [ ]:
iea_regions = pd.read_csv(RAW / 'WEO2025_AnnexA_Free_Dataset_Regions.csv')
peek(iea_regions, 'IEA WEO Regions')

In [ ]:
# Check region labels — these need to align with how we'll map GEM data to regions
for col in ['Region', 'Scenario', 'Category', 'Sub-Category', 'Flow', 'Unit']:
    if col in iea_regions.columns:
        print(f'\n--- {col} ---')
        print(iea_regions[col].value_counts().head(30))

---
## 5. Energy Institute — Statistical Review of World Energy 2025
`Statistical Review of World Energy Narrow format.csv`

Historical energy production/consumption by country and fuel, 1965–2024. Used to cross-check current trends and for demand trend modeling.

In [ ]:
ei = pd.read_csv(RAW / 'Statistical Review of World Energy Narrow format.csv')
peek(ei, 'Energy Institute Statistical Review')

In [ ]:
# Check fuel/variable labels, country names, year range, and units
for col in ['Country', 'Var', 'Variable', 'Fuel', 'Unit', 'Year', 'year']:
    if col in ei.columns:
        print(f'\n--- {col} ---')
        print(ei[col].value_counts().head(30))

# Year range
year_col = next((c for c in ei.columns if 'year' in c.lower()), None)
if year_col:
    print(f'\nYear range: {ei[year_col].min()} – {ei[year_col].max()}')

---
## Summary Checklist

After running all cells, document findings here before moving to `01_data_ingestion.ipynb`:

**GEM GIPT**
- [ ] Confirm main sheet name and header row
- [ ] Fuel/technology column name and unique labels
- [ ] Capacity column name and unit (MW?)
- [ ] Status values (operating, construction, announced, retired…)
- [ ] Coordinate column names

**GEM GNPT**
- [ ] Status progression date columns identified
- [ ] Country names consistent with GIPT?

**IEA World & Regions**
- [ ] Scenario labels match expected (STEPS, NZE, Current Policies)
- [ ] Electricity demand variable name and units (TWh / EJ?)
- [ ] Region labels in Regions file (will need mapping to GEM countries)

**Energy Institute**
- [ ] Variable/fuel label for electricity generation
- [ ] Country name format (ISO? Full name?)
- [ ] Year range confirmed (1965–2024)